# Pelatihan Model Prediksi Sisa Umur Mesin (NASA Turbofan RUL)

Notebook ini melatih model untuk menebak sisa umur mesin pesawat:
1. Model Cepat: LightGBM (pohon keputusan)
2. Model Rangkaian Waktu: PyTorch CNN + BiGRU
3. Model Penantang: PyTorch CNN + BiGRU + Attention + Aturan Rugi Asimetris

Data yang dipakai berasal dari dbt yang sudah bersih dan siap pakai.

## 1. Persiapan Lingkungan dan Instalasi Paket

In [ ]:
!pip install lightgbm optuna onnx onnxruntime onnxscript pyarrow matplotlib -q

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Menggunakan alat:', device)

## 2. Membaca Data Fitur

In [ ]:
df = pd.read_parquet('features_fd001.parquet')
print('Ukuran data:', df.shape)
df.head()

## 3. Pemeriksaan Data Singkat (EDA)

In [ ]:
print('Jumlah mesin pesawat:', df['engine_id'].nunique())
print('Nilai kosong per kolom:', df.isnull().sum().max())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df['target_rul'], bins=20, color='skyblue', edgecolor='black')
plt.title('Penyebaran Target RUL')
plt.xlabel('Sisa Siklus')
plt.ylabel('Frekuensi')

sample_engine = df[df['engine_id'] == 1]
plt.subplot(1, 2, 2)
plt.plot(sample_engine['cycle'], sample_engine['sensor_2_lpc_temp'], color='red')
plt.title('Pola Suhu Mesin #1 Terhadap Siklus')
plt.xlabel('Siklus')
plt.ylabel('Suhu')
plt.tight_layout()
plt.show()

## 4. Pembagian Data Berdasarkan Nomor Mesin

In [ ]:
feature_cols = [
    col for col in df.columns 
    if col not in ['engine_id', 'cycle', 'max_cycle', 'raw_rul', 'target_rul', 'event_timestamp', 'ingested_at']
]
print('Jumlah kolom fitur:', len(feature_cols))

train_engines = df['engine_id'].unique()[:80]
val_engines = df['engine_id'].unique()[80:]

train_df = df[df['engine_id'].isin(train_engines)].copy()
val_df = df[df['engine_id'].isin(val_engines)].copy()

X_train = train_df[feature_cols]
y_train = train_df['target_rul']
X_val = val_df[feature_cols]
y_val = val_df['target_rul']

print('Baris data latih:', len(train_df))
print('Baris data validasi:', len(val_df))

## 5. Rumus Penilaian Khusus NASA

In [ ]:
def calculate_nasa_score(y_true, y_pred):
    diff = y_pred - y_true
    score = 0.0
    for d in diff:
        if d < 0:
            score += (math.exp(-d / 13.0) - 1.0)
        else:
            score += (math.exp(d / 10.0) - 1.0)
    return score

def evaluate_predictions(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    nasa = calculate_nasa_score(y_true.values, y_pred)
    print(f'Hasil Model: {name}')
    print(f'RMSE : {rmse:.2f} siklus')
    print(f'MAE  : {mae:.2f} siklus')
    print(f'Skor NASA : {nasa:.2f}')
    return {'model': name, 'rmse': rmse, 'mae': mae, 'nasa_score': nasa}

## 6. Model 1: LightGBM dan Penyetelan Parameter (Optuna)

In [ ]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15)
print('Parameter Terbaik LightGBM:', study.best_params)

best_lgb = lgb.LGBMRegressor(**study.best_params, random_state=42, verbosity=-1)
best_lgb.fit(X_train, y_train)
lgb_val_preds = best_lgb.predict(X_val)
metrics_lgb = evaluate_predictions('LightGBM (Tuned)', y_val, lgb_val_preds)

## 7. Model 2: Deep Learning Rangkaian Waktu (Sliding Window 30 Siklus)

In [ ]:
scaler = StandardScaler()
scaled_train = scaler.fit_transform(X_train)
scaled_val = scaler.transform(X_val)

train_df_scaled = train_df.copy()
val_df_scaled = val_df.copy()
train_df_scaled[feature_cols] = scaled_train
val_df_scaled[feature_cols] = scaled_val

def create_sliding_windows(data, feature_names, seq_len=30):
    sequences = []
    targets = []
    for engine in data['engine_id'].unique():
        eng_df = data[data['engine_id'] == engine].sort_values('cycle')
        feat_vals = eng_df[feature_names].values
        target_vals = eng_df['target_rul'].values
        
        if len(eng_df) < seq_len:
            pad = np.repeat(feat_vals[0:1, :], seq_len - len(eng_df), axis=0)
            feat_vals = np.vstack([pad, feat_vals])
            target_vals = np.concatenate([np.repeat(target_vals[0], seq_len - len(eng_df)), target_vals])
            
        for i in range(len(feat_vals) - seq_len + 1):
            sequences.append(feat_vals[i:i + seq_len])
            targets.append(target_vals[i + seq_len - 1])
            
    return np.array(sequences, dtype=np.float32), np.array(targets, dtype=np.float32)

X_train_seq, y_train_seq = create_sliding_windows(train_df_scaled, feature_cols, seq_len=30)
X_val_seq, y_val_seq = create_sliding_windows(val_df_scaled, feature_cols, seq_len=30)

class TurbofanDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.sequences)
        
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

train_loader = DataLoader(TurbofanDataset(X_train_seq, y_train_seq), batch_size=64, shuffle=True)
val_loader = DataLoader(TurbofanDataset(X_val_seq, y_val_seq), batch_size=64, shuffle=False)

In [ ]:
class TurbofanBiGRU(nn.Module):
    def __init__(self, num_features, hidden_dim=64, num_layers=2):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(input_size=32, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        x_trans = x.transpose(1, 2)
        conv_out = self.relu(self.conv1(x_trans))
        conv_out = conv_out.transpose(1, 2)
        gru_out, _ = self.gru(conv_out)
        last_hidden = gru_out[:, -1, :]
        out = self.fc(last_hidden)
        return out

deep_model = TurbofanBiGRU(num_features=len(feature_cols)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(deep_model.parameters(), lr=0.001, weight_decay=1e-4)

for epoch in range(1, 21):
    deep_model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(deep_model(bx), by)
        loss.backward()
        optimizer.step()

deep_model.eval()
with torch.no_grad():
    test_x = torch.tensor(X_val_seq, dtype=torch.float32).to(device)
    deep_preds = deep_model(test_x).cpu().numpy().flatten()

metrics_deep = evaluate_predictions('PyTorch CNN-BiGRU', pd.Series(y_val_seq), deep_preds)

## 8. Model 3 (Eksperimen Penantang): Attention BiGRU + Aturan Rugi Asimetris

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        scores = self.attn(x)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights * x, dim=1)
        return context

class AttentionBiGRU(nn.Module):
    def __init__(self, num_features, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(input_size=32, hidden_size=hidden_dim, num_layers=2, batch_first=True, bidirectional=True)
        self.attention = TemporalAttention(hidden_dim * 2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x_trans = x.transpose(1, 2)
        conv_out = self.relu(self.conv1(x_trans))
        conv_out = conv_out.transpose(1, 2)
        gru_out, _ = self.gru(conv_out)
        attn_out = self.attention(gru_out)
        out = self.fc(attn_out)
        return out

class AsymmetricLoss(nn.Module):
    def __init__(self, late_penalty=2.0):
        super().__init__()
        self.late_penalty = late_penalty

    def forward(self, y_pred, y_true):
        diff = y_pred - y_true
        weights = torch.where(diff > 0, self.late_penalty, 1.0)
        return torch.mean(weights * (diff ** 2))

attn_model = AttentionBiGRU(num_features=len(feature_cols)).to(device)
asym_criterion = AsymmetricLoss(late_penalty=2.0)
attn_optimizer = torch.optim.AdamW(attn_model.parameters(), lr=0.001, weight_decay=1e-4)

for epoch in range(1, 26):
    attn_model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        attn_optimizer.zero_grad()
        loss = asym_criterion(attn_model(bx), by)
        loss.backward()
        attn_optimizer.step()

attn_model.eval()
with torch.no_grad():
    test_x = torch.tensor(X_val_seq, dtype=torch.float32).to(device)
    attn_preds = attn_model(test_x).cpu().numpy().flatten()

metrics_attn = evaluate_predictions('Attention-BiGRU (Asymmetric)', pd.Series(y_val_seq), attn_preds)

## 9. Perbandingan Semua Model

In [ ]:
summary = pd.DataFrame([metrics_lgb, metrics_deep, metrics_attn])
display(summary)

plt.figure(figsize=(10, 4))
plt.plot(y_val_seq[:150], label='Nilai Asli', color='black', linewidth=2)
plt.plot(lgb_val_preds[:150], label='LightGBM', color='orange', alpha=0.6)
plt.plot(deep_preds[:150], label='BiGRU Standar', color='blue', alpha=0.5)
plt.plot(attn_preds[:150], label='Attention BiGRU', color='green', linewidth=1.5)
plt.title('Perbandingan Tebakan Sisa Umur (150 Data Pertama)')
plt.xlabel('Urutan Sampel')
plt.ylabel('Sisa Umur (Siklus)')
plt.legend()
plt.show()

## 10. Simpan Model Terbaik ke Format ONNX (Siap Pakai untuk Server)

In [ ]:
winner_model = attn_model if metrics_attn['nasa_score'] < metrics_deep['nasa_score'] else deep_model
winner_model.eval()

dummy_input = torch.randn(1, 30, len(feature_cols), dtype=torch.float32).to(device)
onnx_file_path = 'best_rul_model.onnx'

torch.onnx.export(
    winner_model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input_sequence'],
    output_names=['predicted_rul'],
    dynamic_axes={
        'input_sequence': {0: 'batch_size'},
        'predicted_rul': {0: 'batch_size'}
    }
)

print('Model pemenang berhasil disimpan ke ONNX:', onnx_file_path)